# 🏃 App 1 — Tạo index khuôn mặt & số BIB (Colab/GPU)

Notebook này quét ảnh từ một hoặc nhiều folder Google Drive, sinh **embedding khuôn mặt**
và **đọc số BIB**, rồi xuất file `index.zip` cho từng folder.

**Cách dùng:**
1. **Runtime → Change runtime type → GPU**
2. Upload file `app1_indexer.zip` (chứa code ứng dụng) khi được hỏi
3. Chạy lần lượt 3 cell → giao diện hiện ra
4. Dán link/ID folder Drive → bấm **Bắt đầu quét**
5. Mỗi folder xong → tự tải `index.zip` về máy + lưu lên Drive (tuỳ chọn)

## 1. Upload code & cài thư viện (5–7 phút lần đầu)

In [ ]:
import os, zipfile

WORK   = '/content/app1_indexer'
MARKER = '/content/.deps_installed_v4'

# --- Upload và giải nén zip code ---
if not os.path.exists(WORK):
    from google.colab import files
    print('📦 Hãy upload file app1_indexer.zip:')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall('/content')
    os.remove(zip_name)
    # Xử lý trường hợp zip có thư mục gốc hoặc không
    if not os.path.exists(WORK):
        # Tìm thư mục vừa giải nén
        extracted = [d for d in os.listdir('/content')
                     if os.path.isdir(f'/content/{d}') and d not in
                     ('sample_data', '.config', '.local')]
        for d in extracted:
            if os.path.exists(f'/content/{d}/build_index.py'):
                os.rename(f'/content/{d}', WORK)
                break
        else:
            # File nằm trực tiếp trong /content
            os.makedirs(WORK, exist_ok=True)
            for f in ('build_index.py', 'face_engine.py', 'bib_engine.py',
                      'config.py', 'drive_source.py', 'bib_detector.py'):
                if os.path.exists(f'/content/{f}'):
                    os.rename(f'/content/{f}', f'{WORK}/{f}')

os.chdir(WORK)

NEEDED = ['build_index.py', 'face_engine.py', 'bib_engine.py',
          'config.py', 'drive_source.py', 'bib_detector.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f'Thiếu file: {missing}'
print('✅ Code sẵn sàng tại:', os.getcwd())

# --- Cài deps (chỉ chạy 1 lần mỗi runtime) ---
if os.path.exists(MARKER):
    print('✅ Thư viện đã cài từ trước (skip).')
else:
    !pip install -q \
      "numpy==1.26.4" \
      insightface==0.7.3 onnxruntime-gpu==1.19.2 faiss-cpu==1.8.0 \
      paddleocr==2.7.3 paddlepaddle==2.6.1 ultralytics==8.2.0 \
      opencv-python-headless==4.10.0.84 pandas==2.2.2 pyarrow==16.1.0 \
      google-api-python-client==2.134.0 google-auth==2.30.0 \
      python-dotenv==1.0.1 tqdm==4.66.4 ipywidgets==8.1.3
    !pip install -q --force-reinstall --no-deps "numpy==1.26.4"
    open(MARKER, 'w').close()
    print('\n⚠️  Đã cài xong. KILL RUNTIME để load numpy đúng phiên bản...')
    print('    → Sau khi Colab tự reconnect, CHẠY LẠI CELL NÀY (sẽ skip cài).')
    os.kill(os.getpid(), 9)

!nvidia-smi -L
print('✅ Sẵn sàng.')

## 2. Đăng nhập Google Drive

In [ ]:
import os, sys

WORK = '/content/app1_indexer'
if os.getcwd() != WORK:
    os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default(scopes=['https://www.googleapis.com/auth/drive'])

# Import engines 1 lần
os.environ.setdefault('FACE_CTX_ID', '0')
os.environ.setdefault('ENABLE_BIB', 'true')

from drive_source import DriveSource
from config import IndexerConfig
from build_index import build_index

drive = DriveSource(credentials=creds)
print('✅ Đã đăng nhập Google Drive.')

## 3. Giao diện quét — dán link Drive và bấm chạy

In [ ]:
run_logged(build_index, config, drive=drive, limit=50)

## 4. Chạy toàn bộ folder

Tốn từ vài phút đến vài giờ tuỳ số ảnh. Có thể đóng tab và mở lại — cell vẫn chạy ở phía Colab miễn là runtime chưa bị thu hồi.

In [ ]:
run_logged(build_index, config, drive=drive)

## 5. Tải index về máy

Sau khi tải `index.zip` xong, vào App 2 → trang quản lý → mở dự án → Upload zip này lên là xong.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('index', 'zip', 'data/index')
files.download('index.zip')

## 6. Tải log (chạy khi gặp lỗi)

Tải `run_log.txt` về thư mục `face-app/app1_indexer/` rồi nhắn "đọc log.txt" để Claude xem.

In [ ]:
from google.colab import files
files.download('/content/run_log.txt')